# Data Wrangling

In [1]:
import pandas as pd
import sqlite3
import os
import matplotlib.pyplot as plt
import seaborn as sns

print('### Python version: ' + __import__('sys').version)

### Python version: 3.13.11 (tags/v3.13.11:6278944, Dec  5 2025, 16:26:58) [MSC v.1944 64 bit (AMD64)]


In [2]:
conn = sqlite3.connect('aqi_project.db')

df_demographics = pd.read_sql("SELECT * FROM demographics", conn)
df_income = pd.read_sql("SELECT * FROM income", conn)
df_health = pd.read_sql("SELECT * FROM health", conn)
df_aqi = pd.read_sql("SELECT * FROM air_quality", conn)

df_demographics.head(10)

,id,fips,total_pop,median_age,pop_under_5,pop_under_18,pop_15_to_44,pop_65_and_over
0,1,01003,261608,43.8,12780.0,54250.0,89144.0,59367.0
1,2,01015,116427,39.1,6349.0,24789.0,45839.0,22077.0
2,3,01043,92604,40.4,5666.0,21012.0,33875.0,17836.0
3,4,01049,73122,37.6,3742.0,17149.0,27441.0,13288.0
4,5,01051,91042,40.8,4020.0,19298.0,35643.0,16027.0
5,6,01055,103207,41.3,5846.0,22585.0,36926.0,20914.0
6,7,01069,109366,40.4,6422.0,24757.0,40791.0,21413.0
7,8,01073,664744,37.7,39323.0,150947.0,268748.0,116099.0
8,9,01077,97502,40.8,4705.0,18608.0,38103.0,20934.0
9,10,01081,187847,35.0,9497.0,38241.0,88389.0,26555.0


In [3]:

query = """
    SELECT 
        d.fips,
        d.total_pop,
        d.median_age,
        d.pop_65_and_over,
        i.median_income,
        h.incidence_rate,
        a.days_aqi,
        a.unhealthy_days,
        a.very_unhealthy_days,
        a.hazardous_days,
        a.median_aqi
    FROM demographics d
    JOIN income i ON d.fips = i.fips
    JOIN health h ON d.fips = h.fips
    JOIN air_quality a ON d.fips = a.fips
"""

df_master = pd.read_sql(query, conn)

conn.close()

df_master.head(10)


,fips,total_pop,median_age,pop_65_and_over,median_income,incidence_rate,days_aqi,unhealthy_days,very_unhealthy_days,hazardous_days,median_aqi
0,01003,261608,43.8,59367.0,82501.0,58.2,241,0,0,0,42
1,01049,73122,37.6,13288.0,52763.0,52.7,243,0,0,0,42
2,01051,91042,40.8,16027.0,81846.0,67.7,177,0,0,0,32
3,01055,103207,41.3,20914.0,57959.0,69.4,241,0,0,0,45
4,01073,664744,37.7,116099.0,69363.0,51.5,182,0,0,0,53
5,01089,423355,37.9,68123.0,89086.0,48.7,181,0,0,0,42
6,01097,412339,39.0,74226.0,60379.0,60.0,241,0,0,0,43
7,01101,225894,37.5,38393.0,61159.0,50.8,237,0,0,0,44
8,01103,126084,40.4,22976.0,68686.0,66.6,243,0,0,0,42
9,01117,235969,40.4,42501.0,102265.0,44.6,182,0,0,0,33


### Clean Data

In [4]:

df_master = df_master.drop_duplicates(subset=['fips'], keep='first')

df_master = df_master.dropna(subset=['incidence_rate'])

if df_master['median_income'].isnull().sum() > 0:
    df_master['median_income'] = df_master['median_income'].fillna(df_master['median_income'].median())



### Feature Engineering 

In [5]:

# Calculate the percentage of population that over 65 years old 
df_master['pct_over_65'] = df_master['pop_65_and_over'] / df_master['total_pop']


df_master['unhealthy_days_total'] = (df_master['unhealthy_days'] + 
                                     df_master['very_unhealthy_days'] + 
                                     df_master['hazardous_days'])
# Calculate the percentage of unhealthy days
df_master['pct_unhealthy_air'] = df_master['unhealthy_days_total'] / df_master['days_aqi']

# Categorical Binning -> categorize income into 4 tiers
df_master['income_tier'] = pd.qcut(df_master['median_income'], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])


In [6]:
print(f"Analysis Dataset Shape: {df_master.shape}")

Analysis Dataset Shape: (549, 15)


In [7]:
print("\nData Types:")
print(df_master.dtypes)


Data Types:
fips                      object
total_pop                  int64
median_age               float64
pop_65_and_over          float64
median_income            float64
incidence_rate           float64
days_aqi                   int64
unhealthy_days             int64
very_unhealthy_days        int64
hazardous_days             int64
median_aqi                 int64
pct_over_65              float64
unhealthy_days_total       int64
pct_unhealthy_air        float64
income_tier             category
dtype: object


In [8]:
df_master.head(10)

,fips,total_pop,median_age,pop_65_and_over,median_income,incidence_rate,days_aqi,unhealthy_days,very_unhealthy_days,hazardous_days,median_aqi,pct_over_65,unhealthy_days_total,pct_unhealthy_air,income_tier
0,01003,261608,43.8,59367.0,82501.0,58.2,241,0,0,0,42,0.226931,0,0.0,Med-High
1,01049,73122,37.6,13288.0,52763.0,52.7,243,0,0,0,42,0.181724,0,0.0,Low
2,01051,91042,40.8,16027.0,81846.0,67.7,177,0,0,0,32,0.176040,0,0.0,Med-High
3,01055,103207,41.3,20914.0,57959.0,69.4,241,0,0,0,45,0.202641,0,0.0,Low
4,01073,664744,37.7,116099.0,69363.0,51.5,182,0,0,0,53,0.174652,0,0.0,Med-Low
5,01089,423355,37.9,68123.0,89086.0,48.7,181,0,0,0,42,0.160912,0,0.0,Med-High
6,01097,412339,39.0,74226.0,60379.0,60.0,241,0,0,0,43,0.180012,0,0.0,Low
7,01101,225894,37.5,38393.0,61159.0,50.8,237,0,0,0,44,0.169960,0,0.0,Low
8,01103,126084,40.4,22976.0,68686.0,66.6,243,0,0,0,42,0.182228,0,0.0,Med-Low
9,01117,235969,40.4,42501.0,102265.0,44.6,182,0,0,0,33,0.180113,0,0.0,High


### Produce analysis-ready dataset table and save it to database

In [9]:

conn = sqlite3.connect('aqi_project.db')

df_master.to_sql('analysis_data', conn, if_exists='replace', index=False)

conn.close()
